# Capstone Project
## Image classifier for the SVHN dataset
### Instructions

In this notebook, you will create a neural network that classifies real-world images digits. You will use concepts from throughout this course in building, training, testing, validating and saving your Tensorflow classifier model.

This project is peer-assessed. Within this notebook you will find instructions in each section for how to complete the project. Pay close attention to the instructions as the peer review will be carried out according to a grading rubric that checks key parts of the project instructions. Feel free to add extra cells into the notebook as required.

### How to submit

When you have completed the Capstone project notebook, you will submit a pdf of the notebook for peer review. First ensure that the notebook has been fully executed from beginning to end, and all of the cell outputs are visible. This is important, as the grading rubric depends on the reviewer being able to view the outputs of your notebook. Save the notebook as a pdf (File -> Download as -> PDF via LaTeX). You should then submit this pdf for review.

### Let's get started!

We'll start by running some imports, and loading the dataset. For this project you are free to make further imports throughout the notebook as you wish. 

In [ ]:
import tensorflow as tf
from scipy.io import loadmat

![SVHN overview image](data/svhn_examples.jpg)
For the capstone project, you will use the [SVHN dataset](http://ufldl.stanford.edu/housenumbers/). This is an  image dataset of over 600,000 digit images in all, and is a harder dataset than MNIST as the numbers appear in the context of natural scene images. SVHN is obtained from house numbers in Google Street View images. 

* Y. Netzer, T. Wang, A. Coates, A. Bissacco, B. Wu and A. Y. Ng. "Reading Digits in Natural Images with Unsupervised Feature Learning". NIPS Workshop on Deep Learning and Unsupervised Feature Learning, 2011.

Your goal is to develop an end-to-end workflow for building, training, validating, evaluating and saving a neural network that classifies a real-world image into one of ten classes.

In [ ]:
# Run this cell to load the dataset

train = loadmat('data/train_32x32.mat')
test = loadmat('data/test_32x32.mat')

Both `train` and `test` are dictionaries with keys `X` and `y` for the input images and labels respectively.

## 1. Inspect and preprocess the dataset
* Extract the training and testing images and labels separately from the train and test dictionaries loaded for you.
* Select a random sample of images and corresponding labels from the dataset (at least 10), and display them in a figure.
* Convert the training and test images to grayscale by taking the average across all colour channels for each pixel. _Hint: retain the channel dimension, which will now have size 1._
* Select a random sample of the grayscale images and corresponding labels from the dataset (at least 10), and display them in a figure.

In [ ]:
import numpy as np

train_X = np.rollaxis(train['X'],3,0)    #moving 'sample' dimention on the 1st place
train_y = train['y']%10    #adjusting labels

test_X = np.rollaxis(test['X'],3,0)
test_y = test['y']%10

print('train shape ', train_X.shape)
print('test shape ', test_X.shape)
print(min(train_y),max(train_y))

In [ ]:
import matplotlib.pyplot as plt
import random


%matplotlib inline

indices = random.sample(range(train_X.shape[0]), k=10)

i=0
for idx in indices:
    i=i+1
    ax = plt.subplot(2, 5, i)
    plt.imshow(train_X[idx,:,:,:].astype("uint8"))
    plt.title(train_y[idx])
    plt.axis("off")

plt.show()

In [ ]:
#converting images to grayscalre

gray_train_X = np.mean(train_X, axis=3, keepdims=True)
gray_test_X = np.mean(test_X, axis=3, keepdims=True)

In [ ]:
indices = random.sample(range(train_X.shape[0]), k=10)

i=0
for idx in indices:
    i=i+1
    ax = plt.subplot(2, 5, i)
    plt.imshow(gray_train_X[idx,:,:, 0].astype("uint8"), cmap='gray')
    plt.title(train_y[idx])
    plt.axis("off")

plt.show()

## 2. MLP neural network classifier
* Build an MLP classifier model using the Sequential API. Your model should use only Flatten and Dense layers, with the final layer having a 10-way softmax output. 
* You should design and build the model yourself. Feel free to experiment with different MLP architectures. _Hint: to achieve a reasonable accuracy you won't need to use more than 4 or 5 layers._
* Print out the model summary (using the summary() method)
* Compile and train the model (we recommend a maximum of 30 epochs), making use of both training and validation sets during the training run. 
* Your model should track at least one appropriate metric, and use at least two callbacks during training, one of which should be a ModelCheckpoint callback.
* As a guide, you should aim to achieve a final categorical cross entropy training loss of less than 1.0 (the validation loss might be higher).
* Plot the learning curves for loss vs epoch and accuracy vs epoch for both training and validation sets.
* Compute and display the loss and accuracy of the trained model on the test set.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras import regularizers


model_MLP = Sequential([
    Flatten(input_shape=(32,32,1)),
    Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
    Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
    Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
    Dense(10, activation='softmax'),
])

In [ ]:
model_MLP.summary()

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint


weights_path_mlp='model_MLP'

modelCheckpoint_MLP=tf.keras.callbacks.ModelCheckpoint(
    filepath=weights_path_mlp,
    save_freq='epoch',
    monitor="val_acc",
    save_best_only=True,
    save_weights_only=True
)

earlyStopping=tf.keras.callbacks.EarlyStopping(
    monitor="val_acc",
    min_delta=0,
    patience=4
)

model_MLP.compile(optimizer='adam', 
                  loss='sparse_categorical_crossentropy', 
                  metrics=['acc'])

In [ ]:
#normalizing data
if np.max(gray_train_X) > 1:
    gray_train_X = gray_train_X/255
    gray_test_X = gray_test_X/255

history = model_MLP.fit(gray_train_X, 
                        train_y, 
                        epochs=30, 
                        validation_data=(gray_test_X,test_y), 
                        batch_size=512, 
                        verbose=2, 
                        callbacks=[modelCheckpoint_MLP, earlyStopping])

In [ ]:
plt.plot(history.history['loss'], 'b',label='training loss')
plt.plot(history.history['val_loss'], 'y',label='vadidation loss')
plt.xlabel('epochs')
plt.ylabel('loss')
plt.legend()

plt.show()

plt.plot(history.history['acc'], 'b',label='training accuracy')
plt.plot(history.history['val_acc'], 'y',label='vadidation accuracy')
plt.xlabel('epochs')
plt.ylabel('accuracy')
plt.legend()

plt.show()

In [ ]:
print('loss and accuracy on the test set for MLP: ')
model_MLP.evaluate(gray_test_X,test_y, verbose=2)

## 3. CNN neural network classifier
* Build a CNN classifier model using the Sequential API. Your model should use the Conv2D, MaxPool2D, BatchNormalization, Flatten, Dense and Dropout layers. The final layer should again have a 10-way softmax output. 
* You should design and build the model yourself. Feel free to experiment with different CNN architectures. _Hint: to achieve a reasonable accuracy you won't need to use more than 2 or 3 convolutional layers and 2 fully connected layers.)_
* The CNN model should use fewer trainable parameters than your MLP model.
* Compile and train the model (we recommend a maximum of 30 epochs), making use of both training and validation sets during the training run.
* Your model should track at least one appropriate metric, and use at least two callbacks during training, one of which should be a ModelCheckpoint callback.
* You should aim to beat the MLP model performance with fewer parameters!
* Plot the learning curves for loss vs epoch and accuracy vs epoch for both training and validation sets.
* Compute and display the loss and accuracy of the trained model on the test set.

In [ ]:
from tensorflow.keras.layers import Conv2D, MaxPool2D, BatchNormalization, Dropout

model_CNN = Sequential([
    Conv2D(8,(2,2), activation='relu', kernel_regularizer=regularizers.l2(1e-4), input_shape=(32,32,1)),
    MaxPool2D((2,2)),
    BatchNormalization(),
    Dropout(0.2),
    Conv2D(16,(2,2), activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
    MaxPool2D((2,2)),
    BatchNormalization(),  
    Dropout(0.2),
    Conv2D(32,(2,2), activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])

In [ ]:
model_CNN.summary()

In [ ]:
weights_path_cnn = 'model_CNN'

modelCheckpoint_CNN=tf.keras.callbacks.ModelCheckpoint(
    filepath=weights_path_cnn,
    save_freq='epoch',
    monitor="val_acc",
    save_best_only=True,
    save_weights_only=True
)


model_CNN.compile(optimizer='adam', 
                  loss='sparse_categorical_crossentropy', 
                  metrics=['acc'])

In [ ]:
history = model_CNN.fit(gray_train_X, 
                        train_y, 
                        epochs=5, 
                        validation_data=(gray_test_X,test_y), 
                        batch_size=512, 
                        verbose=2, 
                        callbacks=[modelCheckpoint_CNN, earlyStopping])

In [ ]:
plt.plot(history.history['loss'], 'b',label='training loss')
plt.plot(history.history['val_loss'], 'y',label='vadidation loss')
plt.xlabel('epochs')
plt.ylabel('loss')
plt.legend()

plt.show()

plt.plot(history.history['acc'], 'b',label='training accuracy')
plt.plot(history.history['val_acc'], 'y',label='vadidation accuracy')
plt.xlabel('epochs')
plt.ylabel('accuracy')
plt.legend()

plt.show()

In [ ]:
print('loss and accuracy on the test set for CNN: ')
model_CNN.evaluate(gray_test_X,test_y, verbose=2)

## 4. Get model predictions
* Load the best weights for the MLP and CNN models that you saved during the training run.
* Randomly select 5 images and corresponding labels from the test set and display the images with their labels.
* Alongside the image and label, show each model’s predictive distribution as a bar chart, and the final model prediction given by the label with maximum probability.

In [ ]:
model_CNN.load_weights(weights_path_cnn)
model_MLP.load_weights(weights_path_mlp)

In [ ]:
indices = random.sample(range(test_X.shape[0]), k=5)

for idx in indices:
    fig, (ax1,ax2,ax3) = plt.subplots(1, 3, figsize=(16,2))
    ax1.imshow(test_X[idx,:,:,:].astype("uint8"))
    ax1.set_title(test_y[idx])
    ax1.axis("off")
    
    img = gray_test_X[idx,:,:,:]
    img = img[np.newaxis,...]
    
    CNN_predict=model_CNN.predict(img)[0]
    MLP_predict=model_MLP.predict(img)[0]
    argmax_CNN = np.argmax(CNN_predict)
    argmax_MLP = np.argmax(MLP_predict)

    ax2.set_xticks(range(10))
    ax2.set_title(f'CNN Prediction: {argmax_CNN}')
    ax2.bar(np.arange(len(CNN_predict)), CNN_predict, width=1)

    ax3.set_xticks(range(10))
    ax3.set_title(f'MLP Prediction: {argmax_MLP}')
    ax3.bar(np.arange(len(MLP_predict)), MLP_predict, width=1)
    plt.show()
